# GTEx model building with CLAMP

💡 **Environment:** `clamp-analyses`  

This notebook builds latent variable models from GTEx v8 RNA‑seq TPM data using CLAMP. It automates downloading and preprocessing the GTEx matrix, creates a Filebacked Big Matrix (FBM), computes an SVD to estimate the model dimension, prepares pathway priors, runs CLAMP (base + full), and saves model outputs (B, Z, summaries) and intermediate files. Configuration and paths are controlled via `config.R`.

# CLAMP

## Load libraries

In [1]:
# Create a timestamp to track the start of the analysis
start_time <- Sys.time()
cat("GTEx CLAMP and PLIER analysis started at:", format(start_time), "\n")

GTEx CLAMP and PLIER analysis started at: 2026-04-01 12:54:57 


In [2]:
library(bigstatsr)
library(data.table)
library(dplyr)
library(rsvd)
library(glmnet)
library(Matrix)
library(knitr)
library(here)
library(CLAMP)
library(PCAtools)

source(here("config.R"))

set.seed(123)

MAX_ITER <- 500


Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: Matrix

Loaded glmnet 4.1-10

here() starts at /home/msubirana/Documents/pivlab/clamp-analyses

Loading required package: ggplot2

Loading required package: ggrepel


Attaching package: ‘PCAtools’


The following objects are masked from ‘package:stats’:

    biplot, screeplot




## Output directory

In [3]:
output_data_dir <- config$GTEx$OUTPUT_DIR
dir.create(output_data_dir, showWarnings = FALSE, recursive = TRUE)

output_data_dir

[1] "/home/msubirana/Documents/pivlab/clamp-analyses/output/gtex"

# Settings

In [4]:
block_size <- config$GENERAL$CHUNK_SIZE
N_CORES    <- config$GTEx$N_CORES

## Download GTEx 

In [5]:
url <- config$GTEx$URL
dest_dir <-  config$GTEx$DATASET_FOLDER
dest_gz  <- file.path(dest_dir, basename(url))

if (!file.exists(dest_gz)) {
  dir.create(dest_dir, recursive = TRUE, showWarnings = FALSE)
  download.file(url, dest_gz, mode = "wb")
  message("Downloaded to: ", dest_gz)
} else {
  message("File already exists, skipping download.")
}

File already exists, skipping download.



## Preprocess GTEx data

In [6]:
exprs_path  <- file.path(config$GTEx$DATASET_FOLDER, 'GTEx_Analysis_2017-06-05_v8_RNASeQCv1.1.9_gene_tpm.gct.gz')
output_file <- config$GTEx$DATASET_FILE

if (!file.exists(output_file)) {
  dir.create(dirname(output_file), recursive = TRUE, showWarnings = FALSE)
  exprs_data <- read.table(exprs_path, header = TRUE, sep = "\t", skip = 2, check.names = FALSE)
  saveRDS(exprs_data, config$GTEx$DATASET_FILE)
  message("File successfully written to: ", config$GTEx$DATASET_FILE)
} else {
  message("Output file already exists. Skipping.")
}

# Aggregate in-place by 'description'
gtex <- readRDS(here(config$GTEx$DATASET_FILE))
gtex <- as.data.table(gtex)
aggregated_gtex <- gtex[, lapply(.SD, sum), by = Description, .SDcols = is.numeric]

genes <- aggregated_gtex$Description
samples <- colnames(aggregated_gtex[, -1])
data_mat <- as.matrix(aggregated_gtex[, -1])

Output file already exists. Skipping.



In [7]:
fbm_files <- c(
  file.path(output_data_dir, "FBMgtex.bk"),
  file.path(output_data_dir, "FBMgtex_preproc.bk"),
  file.path(output_data_dir, "FBMgtex_preproc_filtered.bk"),
  file.path(output_data_dir, "FBMgtex_preproc_filtered.rds")
)

for (f in fbm_files) {
  if (file.exists(f)) {
    res <- unlink(f)
    if (identical(res, 0L)) {
      message("Removed existing FBM file: ", f)
    } else {
      warning("Failed to remove: ", f)
    }
  } else {
    message("Not found, skipping: ", f)
  }
}

Removed existing FBM file: /home/msubirana/Documents/pivlab/clamp-analyses/output/gtex/FBMgtex.bk



Removed existing FBM file: /home/msubirana/Documents/pivlab/clamp-analyses/output/gtex/FBMgtex_preproc.bk

Removed existing FBM file: /home/msubirana/Documents/pivlab/clamp-analyses/output/gtex/FBMgtex_preproc_filtered.bk

Not found, skipping: /home/msubirana/Documents/pivlab/clamp-analyses/output/gtex/FBMgtex_preproc_filtered.rds



In [8]:
# Create the FBM
fbm_file <- file.path(output_data_dir, "FBMgtex")
gtexFBM <- FBM(nrow = nrow(data_mat), ncol = ncol(data_mat), backingfile = fbm_file, create_bk = T)

# Populate it with data
n_blocks <- ceiling(nrow(aggregated_gtex) / block_size)

for (i in 1:n_blocks) {
  start_row <- (i-1) * block_size + 1
  end_row <- min(i * block_size, nrow(data_mat))
  
  gtexFBM[start_row:end_row, ] <- as.matrix(data_mat[start_row:end_row, ])
}

# Preprocess and z‑score FBM

prep_gtex <- preprocessCLAMPFBM(
  fbm        = gtexFBM,
  mean_cutoff= config$GTEx$GENES_MEAN_CUTOFF,
  var_cutoff = config$GTEx$GENES_VAR_CUTOFF,
  ncores=N_CORES
)

gtex_fbm_filt <- prep_gtex$fbm_filtered
gtex_rowStats <- prep_gtex$rowStats

Applying log2 transformation

No NA values found



Check overlap genes with GTEx v8 RNA-seq data and TWAS gene list

In [9]:
twas_genes <- read.csv(here('output/creating_twas_gwas_list/gene_list_union_mashr_and_elastic_net_with_phi.csv'))
head(twas_genes)

,Ensembl_ID,Gene_Symbol,Ensembl_ID_w_version,en_Adipose_Subcutaneous,en_Adipose_Visceral_Omentum,en_Adrenal_Gland,en_Artery_Aorta,en_Artery_Coronary,en_Artery_Tibial,en_Brain_Amygdala,⋯,mashr_Skin_Not_Sun_Exposed_Suprapubic,mashr_Skin_Sun_Exposed_Lower_leg,mashr_Small_Intestine_Terminal_Ileum,mashr_Spleen,mashr_Stomach,mashr_Testis,mashr_Thyroid,mashr_Uterus,mashr_Vagina,mashr_Whole_Blood
,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
1,ENSG00000238009,RP11-34P13.7,ENSG00000238009.6,0,0,0,0,0,0,0,⋯,0,0,0,1,0,1,0,0,0,1
2,ENSG00000228463,AP006222.2,ENSG00000228463.9,0,1,1,0,0,0,0,⋯,1,1,0,0,1,1,1,1,1,0
3,ENSG00000237094,RP4-669L17.10,ENSG00000237094.11,0,0,0,0,0,0,0,⋯,1,0,0,0,0,0,1,0,0,0
4,ENSG00000230021,RP5-857K21.4,ENSG00000230021.8,0,0,0,0,0,0,0,⋯,0,0,0,0,0,1,1,0,0,0
5,ENSG00000237491,RP11-206L10.9,ENSG00000237491.8,1,1,1,1,1,1,1,⋯,1,1,1,1,1,1,1,1,1,1
6,ENSG00000177757,FAM87B,ENSG00000177757.2,1,1,1,1,1,1,0,⋯,1,1,1,1,1,1,1,1,1,1


In [10]:
overlap_genes <- intersect(genes[prep_gtex$kept_rows], twas_genes$Gene_Symbol)

percent_overlap <- (length(overlap_genes) / length(twas_genes$Gene_Symbol)) * 100

cat("Number of overlapping genes:", length(overlap_genes), "\n")
cat("Percentage overlap (relative to TWAS genes):", round(percent_overlap, 2), "%\n")

Number of overlapping genes: 16488 
Percentage overlap (relative to TWAS genes): 72.24 %


In [11]:
zscoreCLAMPFBM(gtex_fbm_filt, gtex_rowStats, ncores=N_CORES)
gtex_genes <- genes[prep_gtex$kept_rows]

Applying Z-score transformation



In [12]:
saveRDS(samples, file = file.path(output_data_dir, "gtex_samples.rds"))

In [13]:
saveRDS(gtex_genes, file = file.path(output_data_dir, "gtex_genes.rds"))

In [14]:
saveRDS(gtex_fbm_filt, file = file.path(output_data_dir, "gtex_fbm_filt.rds"))

In [15]:
df_gtex_fbm_filt <- as.data.frame(as.matrix(gtex_fbm_filt[]))

In [16]:
colnames(df_gtex_fbm_filt) <- samples

In [17]:
rownames(df_gtex_fbm_filt) <- gtex_genes

In [18]:
head(df_gtex_fbm_filt)

,GTEX-1117F-0226-SM-5GZZ7,GTEX-1117F-0426-SM-5EGHI,GTEX-1117F-0526-SM-5EGHJ,GTEX-1117F-0626-SM-5N9CS,GTEX-1117F-0726-SM-5GIEN,GTEX-1117F-1326-SM-5EGHH,GTEX-1117F-2426-SM-5EGGH,GTEX-1117F-2526-SM-5GZY6,GTEX-1117F-2826-SM-5GZXL,GTEX-1117F-2926-SM-5GZYI,⋯,GTEX-ZZPU-1126-SM-5N9CW,GTEX-ZZPU-1226-SM-5N9CK,GTEX-ZZPU-1326-SM-5GZWS,GTEX-ZZPU-1426-SM-5GZZ6,GTEX-ZZPU-1826-SM-5E43L,GTEX-ZZPU-2126-SM-5EGIU,GTEX-ZZPU-2226-SM-5EGIV,GTEX-ZZPU-2426-SM-5E44I,GTEX-ZZPU-2626-SM-5E45Y,GTEX-ZZPU-2726-SM-5NQ8O
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
WASH7P,1.44614308,0.1466590,1.15444418,1.8411744,-0.07922225,0.6559264,1.9780371,2.5806060,1.6749832,2.04978755,⋯,-1.0148940,-0.59314323,0.65505136,-0.5693505,-0.48892918,0.2077396965,-0.8016505,-0.28439422,-1.6336384,-0.65163455
RP11-34P13.15,-0.21703290,-1.2491072,-0.71067536,-0.5458638,-0.88156172,-0.4026354,-0.5965898,0.2922380,-0.4949213,0.05590706,⋯,-0.1934056,0.33783682,1.59084836,0.4045636,0.05473854,1.0872298726,0.5077824,-0.19027836,-0.9777920,0.24859783
RP11-34P13.16,0.05000362,-1.2529810,-1.26148519,-0.7514709,-0.77242737,-0.3766090,-0.9801061,0.4096420,-0.3255995,-0.06358077,⋯,0.1692925,0.70250273,1.43810645,0.6769784,0.17310765,1.2674193527,0.8571877,-0.10447332,-1.0448862,0.46041845
RP11-34P13.18,0.80964288,-0.4648762,0.60984294,0.6852051,-1.14329360,-0.5356296,1.4716736,1.0668942,0.1565502,0.90028575,⋯,-1.3100707,-0.67305864,-0.05994370,-1.0821739,-0.70715756,0.0894613755,-1.3990557,-0.65677160,-1.7141016,-0.37942569
AP006222.2,0.60340589,-0.6590192,-0.22788724,-0.3186479,-0.75919490,-0.8022825,-0.8073561,-0.9128508,-0.3943123,-0.90819239,⋯,0.1568485,1.49184552,1.31865074,-0.1807585,0.13219755,2.1816052692,-0.3754781,-0.03196266,-0.2097524,3.06398139
MTND1P23,-0.28701222,0.4584699,-0.04655404,-0.6834497,-0.28493094,0.5112355,-0.4015529,-0.5931863,0.1299671,-0.25562386,⋯,1.0400459,-0.03813723,-0.04317319,-0.1047992,0.06352471,0.0003105984,-0.2519768,-0.01557983,0.3048702,-0.02531932


In [19]:
saveRDS(df_gtex_fbm_filt, file = file.path(output_data_dir, "df_gtex_fbm_filt.rds"))
write.csv(df_gtex_fbm_filt, file = file.path(output_data_dir, "df_gtex_fbm_filt.csv"))

## SVD computation

In [20]:
file.path(output_data_dir, "gtex_svdRes.rds")

[1] "/home/msubirana/Documents/pivlab/clamp-analyses/output/gtex/gtex_svdRes.rds"

In [21]:
if (!file.exists(file.path(output_data_dir, "gtex_svdRes.rds"))) {

  g_fb <- nrow(gtex_fbm_filt)
  samples_fb <- ncol(gtex_fbm_filt)
  SVD_K_gtex <- min(g_fb, samples_fb) - 1
  SVD_K_gtex <- floor(SVD_K_gtex / 4)
  message("Using SVD K = ", SVD_K_gtex)

  gtex_svdRes <- rsvd(
    gtex_fbm_filt[],
    k = SVD_K_gtex
  )

  saveRDS(gtex_svdRes, file = file.path(output_data_dir, "gtex_svdRes.rds"))

} else {
  message("gtex_svdRes already exists, skipping SVD computation.")
}

gtex_svdRes already exists, skipping SVD computation.



## Estimate K for CLAMP

In [22]:
gtex_fbm_filt <- readRDS(file.path(output_data_dir, "gtex_fbm_filt.rds"))
gtex_svdRes <- readRDS(file.path(output_data_dir, "gtex_svdRes.rds"))

In [23]:
n_genes_gtex   <- nrow(gtex_fbm_filt)
n_samples_gtex <- ncol(gtex_fbm_filt)
eigenvalues <- sort(gtex_svdRes$d^2 / (n_samples_gtex - 1), decreasing = TRUE)
noise_gd    <- median(eigenvalues)
CLAMP_K_gtex <- PCAtools::chooseGavishDonoho(
    .dim          = c(n_genes_gtex, n_samples_gtex),
    var.explained = eigenvalues,
    noise         = noise_gd
) * 2
message("Inferred CLAMP K = ", CLAMP_K_gtex)

Inferred CLAMP K = 578



In [24]:
saveRDS(CLAMP_K_gtex, file = file.path(output_data_dir, "CLAMP_K_gtex.rds"))

write.csv(
  as.data.frame(CLAMP_K_gtex),
  file = file.path(output_data_dir, "CLAMP_K_gtex.csv"),
  row.names = TRUE
)

## CLAMPbase

In [25]:
gtex_baseRes <- CLAMPbase(
  Y      = gtex_fbm_filt,
  svdres = gtex_svdRes,
  trace  = TRUE,
  clamp_k = CLAMP_K_gtex
)

gtex_baseRes$Z <- data.frame(gtex_baseRes$Z)
rownames(gtex_baseRes$Z) <- gtex_genes
head(gtex_baseRes$Z)

gtex_baseRes$B <- data.frame(gtex_baseRes$B)
colnames(gtex_baseRes$B) <- samples
head(gtex_baseRes$B)

saveRDS(gtex_baseRes, file = file.path(output_data_dir, "CLAMPbase.rds"))

model_dir <- file.path(output_data_dir, "CLAMPbase")
dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)

B <- gtex_baseRes$B
write.csv(B, file.path(model_dir, "B.csv"))

Z <- gtex_baseRes$Z
write.csv(Z, file.path(model_dir, "Z.csv"))

****

CLAMP k is set to 578

L1 is set to 39.0160308826276

L2 is set to 117.048092647883

Progress 1 / 200 | Bdiff=0.251733, minCor=0.617969

Progress 2 / 200 | Bdiff=0.034415, minCor=0.890862

Progress 3 / 200 | Bdiff=0.015728, minCor=0.961960

Progress 4 / 200 | Bdiff=0.010391, minCor=0.971135

Progress 5 / 200 | Bdiff=0.007700, minCor=0.980103

Progress 6 / 200 | Bdiff=0.006122, minCor=0.985476

Progress 7 / 200 | Bdiff=0.005083, minCor=0.988885

Progress 8 / 200 | Bdiff=0.004338, minCor=0.992227

Progress 9 / 200 | Bdiff=0.003775, minCor=0.992963

Progress 10 / 200 | Bdiff=0.003333, minCor=0.993449

Progress 11 / 200 | Bdiff=0.002980, minCor=0.993906

Progress 12 / 200 | Bdiff=0.002695, minCor=0.994416

Progress 13 / 200 | Bdiff=0.002463, minCor=0.995192

Progress 14 / 200 | Bdiff=0.002273, minCor=0.995463

Progress 15 / 200 | Bdiff=0.002114, minCor=0.995615

Progress 16 / 200 | Bdiff=0.001979, minCor=0.995949

Progress 17 / 200 | Bdiff=0.001863, minCor=0.996295

Progress 18 / 200

,LV1,LV2,LV3,LV4,LV5,LV6,LV7,LV8,LV9,LV10,⋯,LV569,LV570,LV571,LV572,LV573,LV574,LV575,LV576,LV577,LV578
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
WASH7P,0,0.0000000,0.2543043,0,0.0000000,0.0000000,0,0,0,0,⋯,0,0,0,0,0,0.0000000,0,0.0000000,0.0000000,0.1725426
RP11-34P13.15,0,0.0000000,0.5102585,0,0.2970086,0.0000000,0,0,0,0,⋯,0,0,0,0,0,0.0000000,0,0.2189069,0.0000000,0.0000000
RP11-34P13.16,0,0.0000000,0.4467835,0,0.2361539,0.0000000,0,0,0,0,⋯,0,0,0,0,0,0.0000000,0,0.1913230,0.0000000,0.0000000
RP11-34P13.18,0,0.0000000,0.2358132,0,0.0000000,0.0000000,0,0,0,0,⋯,0,0,0,0,0,0.0000000,0,0.0000000,0.0000000,0.0000000
AP006222.2,0,0.3241487,0.2852356,0,0.0000000,0.3402936,0,0,0,0,⋯,0,0,0,0,0,0.3089775,0,0.0000000,0.1156224,0.0000000
MTND1P23,0,0.0000000,0.0000000,0,0.0000000,0.0000000,0,0,0,0,⋯,0,0,0,0,0,0.0000000,0,0.0000000,0.4454087,0.0000000


,GTEX-1117F-0226-SM-5GZZ7,GTEX-1117F-0426-SM-5EGHI,GTEX-1117F-0526-SM-5EGHJ,GTEX-1117F-0626-SM-5N9CS,GTEX-1117F-0726-SM-5GIEN,GTEX-1117F-1326-SM-5EGHH,GTEX-1117F-2426-SM-5EGGH,GTEX-1117F-2526-SM-5GZY6,GTEX-1117F-2826-SM-5GZXL,GTEX-1117F-2926-SM-5GZYI,⋯,GTEX-ZZPU-1126-SM-5N9CW,GTEX-ZZPU-1226-SM-5N9CK,GTEX-ZZPU-1326-SM-5GZWS,GTEX-ZZPU-1426-SM-5GZZ6,GTEX-ZZPU-1826-SM-5E43L,GTEX-ZZPU-2126-SM-5EGIU,GTEX-ZZPU-2226-SM-5EGIV,GTEX-ZZPU-2426-SM-5E44I,GTEX-ZZPU-2626-SM-5E45Y,GTEX-ZZPU-2726-SM-5NQ8O
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
LV1,0.379930039,-0.86183064,0.09460156,0.34833522,-0.04281027,0.31330315,0.46081136,0.23152870,0.33126869,0.2060442138,⋯,-0.17930709,0.13949301,0.18047987,-0.06440647,0.02407654,0.39669961,0.13775249,0.06405662,-1.024633391,0.22122609
LV2,-0.232243612,-0.44885740,-0.24930684,-0.13535532,-0.26378232,-0.29963238,-0.05991158,-0.22019551,-0.32851736,-0.2516580931,⋯,-0.22617794,-0.07635271,-0.18046443,-0.29839277,-0.19436536,-0.06282290,-0.39073459,-0.22727489,-0.429915845,-0.36766869
LV3,0.138718083,-0.34220792,0.03179478,0.06634749,-0.21298096,-0.09525590,0.09319670,0.04570700,0.02104988,0.0003367648,⋯,-0.17335835,-0.02900859,0.07271821,-0.07483807,-0.08528477,-0.09764269,-0.05250344,-0.05878006,-0.365126520,0.03494185
LV4,0.712610067,-0.07431938,0.73649943,0.62949165,0.34272552,0.30307289,0.45900786,-0.66671879,0.34306393,-0.1183601952,⋯,0.33756744,0.02094908,0.04973134,-0.07430095,0.41938970,0.38305186,-1.06787491,0.82724387,-0.145093046,0.48146764
LV5,-0.081014663,-0.20385434,-0.21290309,-0.18318857,-0.24181633,-0.21517602,-0.20003227,1.91800093,-0.07299501,0.6737969162,⋯,-0.19126102,-0.12457114,0.02327963,-0.04211552,-0.27037730,-0.16986960,1.82743054,-0.30168254,-0.255609703,-0.21145126
LV6,0.006941196,0.03677812,-0.04856558,-0.15537427,-0.08615311,-0.09056886,-0.08570583,-0.07603513,-0.05583270,-0.0416951690,⋯,-0.08914492,-0.14023305,-0.06888669,-0.05433584,-0.12825361,-0.08706965,-0.04995089,-0.18160364,0.001195791,-0.10680031


## Prepare pathway priors

In [26]:
gtex_gmtList <- list(
  BP = getGMT("https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=GO_Biological_Process_2025")
)

# prefix each gene-set name with its library to guarantee uniqueness
for(lib in names(gtex_gmtList)) {
  names(gtex_gmtList[[lib]]) <- paste0(lib, "_", names(gtex_gmtList[[lib]]))
}

gtex_pathMat <- gmtListToSparseMat(gtex_gmtList)
gtex_matched <- getMatchedPathwayMat(gtex_pathMat, gtex_genes)
gtex_chatObj <- getChat(gtex_matched)

Auto-detected name: GO_Biological_Process_2025

Using cached file for GO_Biological_Process_2025

There are 12116 genes in the intersection between data and prior

Removing 2020 pathways

Inverting...

done



## CLAMPfull

In [27]:
gtex_fullRes <- CLAMPfull(
    Y = gtex_fbm_filt,
    svdres = gtex_svdRes,
    priorMat = gtex_matched,
    clamp.base.result = gtex_baseRes,
    use_cpp = TRUE,
    trace = TRUE,
    max.iter = MAX_ITER,
    clamp_k = CLAMP_K_gtex
  )

gtex_fullRes$Z <- data.frame(gtex_fullRes$Z)
rownames(gtex_fullRes$Z) <- gtex_genes
head(gtex_fullRes$Z)

gtex_fullRes$B <- data.frame(gtex_fullRes$B)
colnames(gtex_fullRes$B) <- samples
head(gtex_fullRes$B)

gtex_fullRes$summary <- gtex_fullRes$summary %>%
    dplyr::rename(LV = LV_index)  %>% 
    dplyr::mutate(LV = paste0('LV', LV))

colnames(gtex_fullRes$B) <- samples

saveRDS(gtex_fullRes, file = file.path(output_data_dir, "CLAMPfull.rds"))

model_dir <- file.path(output_data_dir, "CLAMPfull")

dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)

B <- gtex_fullRes$B
write.csv(B, file.path(model_dir, "B.csv"))

Z <- gtex_fullRes$Z
write.csv(Z, file.path(model_dir, "Z.csv"))

summary <- gtex_fullRes$summary
write.csv(summary, file.path(model_dir, "summary.csv"))

** CLAMPfull **



using provided CLAMPbase result

CLAMP k is set to 578

L1=39.0160308826276; L2=117.048092647883

Progress 1 / 500 | Bdiff=0.000440

Progress 2 / 500 | Bdiff=0.036679

Progress 3 / 500 | Bdiff=0.014440

Estimated total runtime: ~221.4 min

Progress 4 / 500 | Bdiff=0.009697

Progress 5 / 500 | Bdiff=0.009773

Progress 6 / 500 | Bdiff=0.007915

Progress 7 / 500 | Bdiff=0.007204

Progress 8 / 500 | Bdiff=0.007025

Progress 9 / 500 | Bdiff=0.006632

Progress 10 / 500 | Bdiff=0.006062

Progress 11 / 500 | Bdiff=0.006235

Progress 12 / 500 | Bdiff=0.006334

Progress 13 / 500 | Bdiff=0.006544

Progress 14 / 500 | Bdiff=0.007038

Progress 15 / 500 | Bdiff=0.007317

Progress 16 / 500 | Bdiff=0.007271

Progress 17 / 500 | Bdiff=0.006952

Progress 18 / 500 | Bdiff=0.006688

Progress 19 / 500 | Bdiff=0.007440

Progress 20 / 500 | Bdiff=0.007513

Progress 21 / 500 | Bdiff=0.007584

Progress 22 / 500 | Bdiff=0.007101

Progress 23 / 500 | Bdiff=0.006851

Progress 24 / 500 | Bdiff=0.006363

Progress 2

,LV1,LV2,LV3,LV4,LV5,LV6,LV7,LV8,LV9,LV10,⋯,LV569,LV570,LV571,LV572,LV573,LV574,LV575,LV576,LV577,LV578
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
WASH7P,0,0.0000000,0,0.0000000,0.1157278,0.0000000,0,0,0,0.2265504,⋯,0,0.000000,0.0000000,0,0.0000000,0.0000000,0.0000000,0,0,0
RP11-34P13.15,0,0.1685128,0,0.0000000,0.0000000,0.0000000,0,0,0,0.0000000,⋯,0,1.167474,0.0000000,0,0.0000000,0.3576457,0.0000000,0,0,0
RP11-34P13.16,0,0.0000000,0,0.0000000,0.0000000,0.0000000,0,0,0,0.0000000,⋯,0,1.130827,0.0000000,0,0.1029751,0.0000000,0.0000000,0,0,0
RP11-34P13.18,0,0.0000000,0,0.0000000,0.2270677,0.0000000,0,0,0,0.0000000,⋯,0,0.000000,0.0000000,0,0.0000000,0.0000000,0.0000000,0,0,0
AP006222.2,0,0.1483829,0,0.1072884,0.1620368,0.2668122,0,0,0,0.0000000,⋯,0,1.101877,0.0000000,0,0.0000000,0.0000000,0.3715002,0,0,0
MTND1P23,0,0.0000000,0,0.0000000,0.0000000,0.0000000,0,0,0,0.0000000,⋯,0,0.000000,0.1224662,0,0.0000000,0.0000000,0.0000000,0,0,0


,GTEX-1117F-0226-SM-5GZZ7,GTEX-1117F-0426-SM-5EGHI,GTEX-1117F-0526-SM-5EGHJ,GTEX-1117F-0626-SM-5N9CS,GTEX-1117F-0726-SM-5GIEN,GTEX-1117F-1326-SM-5EGHH,GTEX-1117F-2426-SM-5EGGH,GTEX-1117F-2526-SM-5GZY6,GTEX-1117F-2826-SM-5GZXL,GTEX-1117F-2926-SM-5GZYI,⋯,GTEX-ZZPU-1126-SM-5N9CW,GTEX-ZZPU-1226-SM-5N9CK,GTEX-ZZPU-1326-SM-5GZWS,GTEX-ZZPU-1426-SM-5GZZ6,GTEX-ZZPU-1826-SM-5E43L,GTEX-ZZPU-2126-SM-5EGIU,GTEX-ZZPU-2226-SM-5EGIV,GTEX-ZZPU-2426-SM-5E44I,GTEX-ZZPU-2626-SM-5E45Y,GTEX-ZZPU-2726-SM-5NQ8O
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
LV1,0.03821956,-0.03571818,0.05802942,0.046666791,0.014034944,-0.002800468,0.03697084,0.029602834,0.075848377,0.07010908,⋯,-0.074388157,0.060692617,0.008792729,0.0003461299,-0.06470433,0.015186520,0.001465893,-0.066104370,-0.14539397,0.006629273
LV2,-0.17291288,-0.14243600,-0.17739501,-0.129253648,-0.142321292,-0.174625053,-0.17258179,-0.079781836,-0.184006127,-0.09276065,⋯,-0.140227208,-0.095136054,-0.129895561,-0.1099698167,-0.11440356,-0.178206564,-0.090774709,-0.173436677,-0.17577256,-0.177160328
LV3,-0.05296651,-0.11971361,0.04530590,-0.177611809,-0.004370614,-0.013131012,0.07196506,-0.072757031,-0.003388647,0.05414044,⋯,0.031744709,1.396568815,-0.218557034,0.1230302709,-0.09118146,0.613275625,0.111562594,0.140790502,-0.07357239,-0.006681761
LV4,0.68494019,0.19415146,0.25898676,0.042776845,0.297041671,0.270592102,0.07172505,-0.321013072,0.272884541,0.10510244,⋯,0.162523124,-0.028582823,0.097821213,0.0328774333,-0.02090102,-0.004998039,-0.422042417,-0.002357224,-0.12670400,0.265592190
LV5,0.23819807,0.20642420,0.08247007,0.006713620,0.030963011,0.042210785,-0.15837729,-0.226135817,0.080163538,-0.18355058,⋯,0.009347065,-0.006412812,-0.007506400,0.1600134325,-0.02819102,-0.011097472,-0.354041301,-0.074877808,0.05885928,0.121228465
LV6,-0.14807144,-0.03968658,-0.06460291,-0.009901864,0.035778041,0.038168313,0.01171858,0.003994514,0.057739083,0.03428057,⋯,0.024499994,-0.060479693,-0.112845659,-0.0307300023,-0.01383021,-0.098464529,0.037195757,0.067035430,-0.05052974,0.022353486
